In [11]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
panel = pd.read_csv("../data/processed/merged_panel.csv")
panel.head()

,State,Year,Area_1000ha,Prod_1000tons,Yield_t_ha,Kharif_Rain_mm
0,Andhra Pradesh,2000,2694.75,8040.68,2.983832,832.93
1,Andhra Pradesh,2001,2515.37,7823.69,3.110354,509.79
2,Andhra Pradesh,2002,1867.00,5315.00,2.846813,380.95
3,Andhra Pradesh,2003,1957.32,6054.11,3.093061,501.31
4,Andhra Pradesh,2004,2227.69,7392.67,3.318536,490.10


In [12]:
panel["Rain_Mean_State"] = panel.groupby("State")["Kharif_Rain_mm"].transform("mean")

panel["Rain_Anomaly"] = (
    panel["Kharif_Rain_mm"] - panel["Rain_Mean_State"]
)

In [13]:
# State mean rainfall
panel["Rain_Mean_State"] = panel.groupby("State")["Kharif_Rain_mm"].transform("mean")

# Rainfall anomaly
panel["Rain_Anomaly"] = (
    panel["Kharif_Rain_mm"] - panel["Rain_Mean_State"]
)

# State rainfall std
panel["Rain_Std_State"] = panel.groupby("State")["Kharif_Rain_mm"].transform("std")

# Z-score rainfall
panel["Rain_Z"] = panel["Rain_Anomaly"] / panel["Rain_Std_State"]

In [14]:
panel_model = panel.dropna(subset=["Yield_t_ha", "Rain_Z"])
panel_model.head()

,State,Year,Area_1000ha,Prod_1000tons,Yield_t_ha,Kharif_Rain_mm,Rain_Mean_State,Rain_Anomaly,Rain_Std_State,Rain_Z
0,Andhra Pradesh,2000,2694.75,8040.68,2.983832,832.93,649.826,183.104,145.792448,1.255922
1,Andhra Pradesh,2001,2515.37,7823.69,3.110354,509.79,649.826,-140.036,145.792448,-0.960516
2,Andhra Pradesh,2002,1867.00,5315.00,2.846813,380.95,649.826,-268.876,145.792448,-1.844238
3,Andhra Pradesh,2003,1957.32,6054.11,3.093061,501.31,649.826,-148.516,145.792448,-1.018681
4,Andhra Pradesh,2004,2227.69,7392.67,3.318536,490.10,649.826,-159.726,145.792448,-1.095571


In [15]:
model = smf.ols(
    "Yield_t_ha ~ Rain_Z + C(State)",
    data=panel_model
).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             Yield_t_ha   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.798
Method:                 Least Squares   F-statistic:                     79.79
Date:                Mon, 02 Mar 2026   Prob (F-statistic):          2.30e-116
Time:                        16:54:40   Log-Likelihood:                -146.34
No. Observations:                 380   AIC:                             332.7
Df Residuals:                     360   BIC:                             411.5
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [16]:
print("Rain_Z coefficient:", model.params["Rain_Z"])
print("Rain_Z p-value:", model.pvalues["Rain_Z"])
print("Model R²:", model.rsquared)

Rain_Z coefficient: 0.11978212795176181
Rain_Z p-value: 1.3125983541529882e-09
Model R²: 0.8080960385659655


In [17]:
X = panel_model[["Rain_Z"]]
y = panel_model["Yield_t_ha"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
rmse_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_test)
    
    r2_scores.append(r2_score(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))

print("Linear Mean R²:", np.mean(r2_scores))
print("Linear Mean RMSE:", np.mean(rmse_scores))

Linear Mean R²: -0.01505584859150233
Linear Mean RMSE: 0.8087697705086792


In [18]:
rf_r2 = []
rf_rmse = []

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    rf_r2.append(r2_score(y_test, y_pred))
    rf_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))

print("RF Mean R²:", np.mean(rf_r2))
print("RF Mean RMSE:", np.mean(rf_rmse))

RF Mean R²: -0.4415429083629873
RF Mean RMSE: 0.9569069484382968


In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = panel_model[["Rain_Z", "State"]]
y = panel_model["Yield_t_ha"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), ["State"])
    ],
    remainder="passthrough"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
rmse_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    r2_scores.append(r2_score(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))

print("FE Linear Mean R²:", np.mean(r2_scores))
print("FE Linear Mean RMSE:", np.mean(rmse_scores))

FE Linear Mean R²: 0.7825078687164044
FE Linear Mean RMSE: 0.37439956412605097


In [20]:
state_summary = panel_model.groupby("State").agg(
    Rain_Variability=("Kharif_Rain_mm", "std"),
    Yield_Variability=("Yield_t_ha", "std"),
    Mean_Yield=("Yield_t_ha", "mean")
).reset_index()

state_summary.head()

,State,Rain_Variability,Yield_Variability,Mean_Yield
0,Andhra Pradesh,145.792448,0.326526,3.294837
1,Assam,139.786955,0.313235,1.812256
2,Bihar,205.048716,0.561128,1.762532
3,Chhattisgarh,210.515294,0.470891,1.528034
4,Gujarat,275.593063,0.371694,1.928432


In [21]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = state_summary[["Rain_Variability", "Yield_Variability", "Mean_Yield"]]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42)
state_summary["Cluster"] = kmeans.fit_predict(scaled_features)

state_summary.sort_values("Cluster")

,State,Rain_Variability,Yield_Variability,Mean_Yield,Cluster
1,Assam,139.786955,0.313235,1.812256,0
2,Bihar,205.048716,0.561128,1.762532,0
3,Chhattisgarh,210.515294,0.470891,1.528034,0
13,Rajasthan,149.181896,0.459094,1.791628,0
7,Jharkhand,205.021144,0.491263,1.676430,0
10,Madhya Pradesh,275.856548,0.637906,1.311965,0
9,Kerala,310.955739,0.290373,2.522302,1
16,Uttar Pradesh,226.990323,0.292368,2.225233,1
11,Maharashtra,132.808133,0.256133,1.810435,1
18,West Bengal,229.776044,0.206216,2.684843,1
